# Finance Notebook: Market Risk VaR Backtesting (Expanded)

This notebook builds a review-grade market risk workflow for daily VaR monitoring and validation.

What is included:
- synthetic multi-asset return simulation with regime shifts,
- return diagnostics (tail behavior and volatility state changes),
- rolling VaR models (historical, parametric-normal, EWMA),
- backtesting with exception analysis,
- Kupiec and Christoffersen statistical tests,
- stressed-risk interpretation for governance.

## 0) Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2, kurtosis, norm, skew

np.random.seed(2026)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1) Simulate multi-asset returns with regime shifts

In [ ]:
n_days = 1500
n_assets = 5
portfolio_notional = 12_000_000

base_cov = np.array(
    [
        [0.00010, 0.00003, 0.00002, 0.00001, 0.00002],
        [0.00003, 0.00012, 0.00002, 0.00002, 0.00003],
        [0.00002, 0.00002, 0.00009, 0.00003, 0.00001],
        [0.00001, 0.00002, 0.00003, 0.00011, 0.00002],
        [0.00002, 0.00003, 0.00001, 0.00002, 0.00010],
    ]
)
weights = np.array([0.28, 0.22, 0.18, 0.17, 0.15])

rets = []
for t in range(n_days):
    if t < 600:
        vol_scale = 1.0
    elif t < 980:
        vol_scale = 1.65
    elif t < 1250:
        vol_scale = 1.25
    else:
        vol_scale = 0.95

    # occasional shock days
    shock = np.random.binomial(1, 0.015)
    shock_scale = 2.5 if shock == 1 else 1.0

    r = np.random.multivariate_normal(mean=np.zeros(n_assets), cov=base_cov * vol_scale * shock_scale)
    rets.append(r)

ret = pd.DataFrame(rets, columns=[f"asset_{i+1}" for i in range(n_assets)])
ret["portfolio_ret"] = ret.to_numpy() @ weights
ret["pnl"] = portfolio_notional * ret["portfolio_ret"]
ret["loss"] = -ret["pnl"]
ret.head()

## 2) Return diagnostics

In [ ]:
stats = pd.Series(
    {
        "mean": ret["portfolio_ret"].mean(),
        "std": ret["portfolio_ret"].std(ddof=1),
        "skew": skew(ret["portfolio_ret"]),
        "excess_kurtosis": kurtosis(ret["portfolio_ret"], fisher=True),
        "p01": ret["portfolio_ret"].quantile(0.01),
        "p99": ret["portfolio_ret"].quantile(0.99),
    }
)
stats

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

sns.histplot(ret["portfolio_ret"], bins=70, kde=True, ax=ax[0], color="#4C72B0")
ax[0].set_title("Portfolio Return Distribution")
ax[0].set_xlabel("Daily return")

rolling_vol = ret["portfolio_ret"].rolling(30).std(ddof=1)
ax[1].plot(rolling_vol, color="#C44E52")
ax[1].set_title("30-Day Rolling Volatility")
ax[1].set_xlabel("Day")
ax[1].set_ylabel("Volatility")

plt.tight_layout()
plt.show()

## 3) Rolling VaR models (99%)

In [ ]:
alpha = 0.99
tail_p = 1 - alpha
window = 250
z = norm.ppf(alpha)

# Historical simulation VaR
ret["var_hist_99"] = (
    ret["portfolio_ret"]
    .rolling(window)
    .apply(lambda x: -np.quantile(x, tail_p), raw=True)
    * portfolio_notional
)

# Parametric normal VaR
mu = ret["portfolio_ret"].rolling(window).mean()
sig = ret["portfolio_ret"].rolling(window).std(ddof=1)
ret["var_param_99"] = (-(mu - z * sig) * portfolio_notional)

# EWMA VaR (RiskMetrics-style)
lambda_ = 0.94
ewma_var = np.full(len(ret), np.nan)
seed_var = ret["portfolio_ret"].iloc[:window].var(ddof=1)
ewma_var[window] = seed_var
for t in range(window + 1, len(ret)):
    prev_r = ret["portfolio_ret"].iloc[t - 1]
    ewma_var[t] = lambda_ * ewma_var[t - 1] + (1 - lambda_) * (prev_r**2)

ret["var_ewma_99"] = z * np.sqrt(ewma_var) * portfolio_notional
ret[["var_hist_99", "var_param_99", "var_ewma_99"]].dropna().head()

## 4) Backtest exception analysis

In [ ]:
bt = ret.dropna().copy()

for col in ["var_hist_99", "var_param_99", "var_ewma_99"]:
    bt[f"exc_{col}"] = (bt["loss"] > bt[col]).astype(int)

summary = []
for name, exc_col, var_col in [
    ("Historical", "exc_var_hist_99", "var_hist_99"),
    ("Parametric", "exc_var_param_99", "var_param_99"),
    ("EWMA", "exc_var_ewma_99", "var_ewma_99"),
]:
    n = len(bt)
    x = int(bt[exc_col].sum())
    summary.append(
        {
            "model": name,
            "observations": n,
            "exceptions": x,
            "expected_exceptions": tail_p * n,
            "exception_rate": x / n,
            "avg_var": bt[var_col].mean(),
        }
    )

summary_df = pd.DataFrame(summary).sort_values("exceptions")
summary_df

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(summary_df["model"], summary_df["exceptions"], color=["#4C72B0", "#55A868", "#C44E52"])
plt.axhline(summary_df["expected_exceptions"].iloc[0], linestyle="--", color="black", label="Expected")
plt.title("99% VaR Exception Counts")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

## 5) Statistical validation: Kupiec and Christoffersen

In [ ]:
def kupiec_pof(n: int, x: int, p: float):
    pi = np.clip(x / max(n, 1), 1e-12, 1 - 1e-12)
    ll_h0 = (n - x) * np.log(1 - p) + x * np.log(p)
    ll_h1 = (n - x) * np.log(1 - pi) + x * np.log(pi)
    lr = -2 * (ll_h0 - ll_h1)
    pv = 1 - chi2.cdf(lr, 1)
    return lr, pv


def christoffersen_independence(exceptions: np.ndarray):
    e = exceptions.astype(int)
    n00 = n01 = n10 = n11 = 0
    for i in range(1, len(e)):
        prev, cur = e[i - 1], e[i]
        if prev == 0 and cur == 0:
            n00 += 1
        elif prev == 0 and cur == 1:
            n01 += 1
        elif prev == 1 and cur == 0:
            n10 += 1
        else:
            n11 += 1

    pi0 = n01 / max(n00 + n01, 1)
    pi1 = n11 / max(n10 + n11, 1)
    pi = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)

    pi0 = np.clip(pi0, 1e-12, 1 - 1e-12)
    pi1 = np.clip(pi1, 1e-12, 1 - 1e-12)
    pi = np.clip(pi, 1e-12, 1 - 1e-12)

    ll_ind = (n00 + n10) * np.log(1 - pi) + (n01 + n11) * np.log(pi)
    ll_dep = n00 * np.log(1 - pi0) + n01 * np.log(pi0) + n10 * np.log(1 - pi1) + n11 * np.log(pi1)

    lr_ind = -2 * (ll_ind - ll_dep)
    pv_ind = 1 - chi2.cdf(lr_ind, 1)
    return lr_ind, pv_ind

rows = []
for name, exc_col in [
    ("Historical", "exc_var_hist_99"),
    ("Parametric", "exc_var_param_99"),
    ("EWMA", "exc_var_ewma_99"),
]:
    e = bt[exc_col].to_numpy()
    n = len(e)
    x = int(e.sum())

    lr_pof, pv_pof = kupiec_pof(n, x, tail_p)
    lr_ind, pv_ind = christoffersen_independence(e)
    lr_cc = lr_pof + lr_ind
    pv_cc = 1 - chi2.cdf(lr_cc, 2)

    rows.append(
        {
            "model": name,
            "kupiec_lr": lr_pof,
            "kupiec_p": pv_pof,
            "christoffersen_lr": lr_ind,
            "christoffersen_p": pv_ind,
            "conditional_coverage_lr": lr_cc,
            "conditional_coverage_p": pv_cc,
        }
    )

test_df = pd.DataFrame(rows)
test_df

## 6) Visualization of losses vs VaR lines

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)

for i, (name, var_col, color) in enumerate(
    [
        ("Historical", "var_hist_99", "#4C72B0"),
        ("Parametric", "var_param_99", "#55A868"),
        ("EWMA", "var_ewma_99", "#C44E52"),
    ]
):
    ax[i].plot(bt.index, bt["loss"], label="Loss", color="black", linewidth=0.8)
    ax[i].plot(bt.index, bt[var_col], label=f"{name} VaR 99%", color=color, linewidth=1.0)
    ax[i].set_title(f"Loss vs {name} VaR")
    ax[i].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 7) Regime-wise model robustness

In [ ]:
# Define broad regimes by date index blocks from simulation design
regime = np.where(bt.index < 600, "normal_1", np.where(bt.index < 980, "stress", np.where(bt.index < 1250, "elevated", "normal_2")))
bt["regime"] = regime

reg_rows = []
for rg, sub in bt.groupby("regime"):
    for name, exc_col in [
        ("Historical", "exc_var_hist_99"),
        ("Parametric", "exc_var_param_99"),
        ("EWMA", "exc_var_ewma_99"),
    ]:
        reg_rows.append(
            {
                "regime": rg,
                "model": name,
                "n": len(sub),
                "exceptions": int(sub[exc_col].sum()),
                "exception_rate": sub[exc_col].mean(),
            }
        )

regime_perf = pd.DataFrame(reg_rows)
regime_perf

In [ ]:
plt.figure(figsize=(9, 4.8))
sns.barplot(data=regime_perf, x="regime", y="exception_rate", hue="model", palette="Set2")
plt.axhline(tail_p, linestyle="--", color="black", label="Target exception rate")
plt.title("Exception Rates by Volatility Regime")
plt.ylabel("Exception rate")
plt.tight_layout()
plt.show()

## 8) Stressed capital interpretation

In [ ]:
capital_view = summary_df.merge(test_df, on="model")
capital_view["model_rank_by_exceptions"] = capital_view["exceptions"].rank(method="min")
capital_view["passes_kupiec_5pct"] = capital_view["kupiec_p"] > 0.05
capital_view["passes_cc_5pct"] = capital_view["conditional_coverage_p"] > 0.05

capital_view[[
    "model",
    "avg_var",
    "exceptions",
    "exception_rate",
    "kupiec_p",
    "conditional_coverage_p",
    "passes_kupiec_5pct",
    "passes_cc_5pct",
]]

In [ ]:
best_model = capital_view.sort_values(["passes_cc_5pct", "exceptions", "avg_var"], ascending=[False, True, True]).iloc[0]

print("Recommended production VaR candidate:")
print(f"- Model: {best_model['model']}")
print(f"- Average daily VaR: {best_model['avg_var']:.0f}")
print(f"- Exceptions: {int(best_model['exceptions'])} of {int(best_model['observations'])}")
print(f"- Kupiec p-value: {best_model['kupiec_p']:.4f}")
print(f"- Conditional coverage p-value: {best_model['conditional_coverage_p']:.4f}")

## 9) Final summary

- VaR governance should combine model conservatism and statistical validity.
- Backtests can pass in calm regimes but fail clustering tests during stress.
- EWMA often adapts faster to volatility shifts, but model selection should remain data-driven.
- This notebook provides a transparent basis for model-risk committee review.